# Pillar3d **mE2lam02** — E2 dilution-match: full 27.6k at λ=0.02 (per-correction weight = the bar recipe)

mC recipe (warm-start pillar3b_epoch_20, lr 5e-5, T 0.5, aux_T 0.5, weighted, warmup 0.5),
**corpus `corrections_corpus_mm05.pt`, λ=0.02**, 3 epochs (~2h A100). Bar (local 1k eval_policy,
775000..775999, batch 256): **16,738** = 13.8k mC-ep2; mC27k-ep2 scored 14,943.
Checkpoints → Drive as `pillar3d_mE2lam02_epoch_*.pt` (fresh names).

**Upload to `MyDrive/alphatrain/`:** `colorlines_pillar3d_v2.tar.gz`, **`corrections_corpus_mm05.pt`**,
`v13_pillar3a.pt.gz`, `pillar3b_epoch_20.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time
DRIVE = '/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True); os.makedirs('/content/crisis', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3b_epoch_20.pt', '/content/alphatrain/data/pillar3b_epoch_20.pt')
shutil.copy(f'{DRIVE}/corrections_corpus_mm05.pt', '/content/crisis/corrections_corpus_mm05.pt')
import torch
_c = torch.load('/content/crisis/corrections_corpus_mm05.pt')
print('corpus anchors', _c['boards'].shape[0], _c['_stats'])
assert 26000 < _c['boards'].shape[0] < 30000 and _c['_stats']['min_margin'] >= 0.05, \
    "unexpected corpus — upload corrections_corpus_mm05.pt"

t0 = time.time()
!cp {DRIVE}/v13_pillar3a.pt.gz /content/v13_pillar3a.pt.gz
gz_size = os.path.getsize('/content/v13_pillar3a.pt.gz')
EXPECTED_GZ = 642_409_267
assert gz_size == EXPECTED_GZ, (
    f'.gz on Drive is truncated! got {gz_size} expected {EXPECTED_GZ}. Re-upload.')
!gunzip -t /content/v13_pillar3a.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/v13_pillar3a.pt.gz > /content/alphatrain/data/v13_pillar3a.pt
pt_size = os.path.getsize('/content/alphatrain/data/v13_pillar3a.pt')
EXPECTED_PT = 5_433_958_495
assert pt_size == EXPECTED_PT, f'.pt size wrong! got {pt_size} expected {EXPECTED_PT}.'
print(f'V13 tensor: {pt_size/1e9:.2f} GB ({time.time()-t0:.0f}s)')
!rm /content/v13_pillar3a.pt.gz
%cd /content
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Train mE2lam02  (~40 min/epoch A100)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v13_pillar3a.pt --amp --compile \
    --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
    --epochs 3 --batch-size 32768 --lr 5e-5 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --aux-corrections-corpus crisis/corrections_corpus_mm05.pt --aux-weighted \
    --aux-lambda 0.02 --aux-target-temperature 0.5 \
    --aux-holdout-frac 0.15 --aux-split-seed 0 \
    --aux-batch-size 256 --aux-warmup-epochs 0.5 --aux-preflight-every 200 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3d_mE2lam02_best.pt \
    --save-dir /content/checkpoints/pillar3d_mE2lam02 2>&1 | tee /content/pillar3d_mE2lam02_train.log

In [ ]:
import glob, shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar3d_mE2lam02/epoch_*.pt')):
    dst = f'{DRIVE}/pillar3d_mE2lam02_{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('copied', os.path.basename(f))
shutil.copy('/content/pillar3d_mE2lam02_train.log', f'{DRIVE}/pillar3d_mE2lam02_train.log')

## After training (local, M5)
```
PYTHONPATH=. python -m scripts.eval_policy --model pillar3d_mE2lam02_epoch_2.pt \
    --seed-start 775000 --seed-end 775999 --device mps --batch 256
```
vs bar **16,738** (13.8k mC-ep2) and mC27k-ep2's **14,943**.